# SignBridge — Train ISL & BSL Models (Google Colab, free tier)

This notebook trains **new** sign-language recognition models for:

- 🇮🇳 **Indian Sign Language (ISL)** — two-handed fingerspelling alphabet
- 🇬🇧 **British Sign Language (BSL)** — two-handed fingerspelling alphabet

and exports them in **exactly** the file format SignBridge already expects, so you can drop the output straight into:

```
signBridge/react-app/public/model/isl/
signBridge/react-app/public/model/bsl/
```

## Your ASL model is never touched

Nothing here reads, writes, or retrains anything in `public/model/asl/`. ASL keeps its own `single-hand-78` pipeline and its own files, completely untouched. This notebook only ever writes to `isl/` and `bsl/` output folders.

## Why this plugs into your existing app with (almost) zero code changes

Your `src/utils/normalize.js` already defines a `two-hand-126` feature pipeline (126 = 2 hands × 63 wrist-normalized landmark values), and `src/config/languages.js` already points the `isl` and `bsl` entries at it with `pipeline: 'two-hand-126'`. That pipeline exists in your codebase today but has never had a trained model behind it — it's marked `status: 'comingSoon'`.

This notebook trains models **against that exact same 126-feature pipeline**, re-implemented in Python line-for-line, so the JavaScript that runs in the browser and the Python that trains the model agree on every number, byte for byte. That means the only integration steps are:

1. Copy 4 output files per language into `public/model/isl/` and `public/model/bsl/`
2. Flip `status: 'comingSoon'` → `status: 'available'` for those two entries in `src/config/languages.js`
3. Nothing else — `useSignModel.js`, `normalize.js`, `useMediaPipe.js`, and the screens already know how to load and run any language marked available.

(Full integration checklist is at the bottom of this notebook.)

## Runtime

Free Colab **CPU** is enough for all of this — no GPU required. Training itself is a small MLP on landmark vectors (seconds per epoch). The only moderately slow step is running MediaPipe Hands over the BSL image dataset once, which takes a few minutes on CPU.

## Before you start

You need a free Kaggle account and an API token (`kaggle.json`) so this notebook can download datasets programmatically. Instructions are in the next section.


## 1. Install dependencies

In [ ]:
# Colab's preinstalled TensorFlow is fine — we just add the extra packages we need.
# If pip reports a dependency conflict with tensorflow/tensorflowjs after this cell,
# go to Runtime > Restart runtime, then continue from the NEXT cell (no need to
# re-run this install cell after restarting).
# IMPORTANT: Using mediapipe<0.11.0 to ensure mp.solutions.hands API compatibility
!pip -q install kaggle "mediapipe<0.11.0" opencv-python-headless tensorflowjs scikit-learn --upgrade

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')), "(fine either way — not required)")


## 2. Kaggle API credentials

1. Go to https://www.kaggle.com/settings → **API** → **Create New Token**. This downloads `kaggle.json`.
2. Run the cell below and upload that file when prompted.


In [ ]:
from google.colab import files
import os, shutil

uploaded = files.upload()  # choose the kaggle.json you just downloaded
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle credentials installed.")


## 3. Download datasets

- **ISL** — [Indian Sign Language Hand Landmarks Dataset](https://www.kaggle.com/datasets/eraakash/indian-sign-language-hand-landmarks-dataset): landmarks are *already extracted* (~50k samples, A–Z), so there's no image processing needed — this is the fastest, most reliable option currently on Kaggle for ISL.
- **BSL** — [BSL Fingerspelling Dataset](https://www.kaggle.com/datasets/alifsathar/bsl-fingerspelling-dataset): labeled two-handed A–Z images. No landmark-only BSL dataset exists publicly yet, so we run MediaPipe Hands over these images ourselves, later in this notebook.

If you find a dataset you like better, just swap the slug in `kaggle datasets download -d <slug>` below. The parsing cells further down print diagnostics (column names, detected labels) so you can quickly tell if a different dataset's layout needs adjusting.


In [ ]:
import os

os.makedirs('/content/data/isl_raw', exist_ok=True)
os.makedirs('/content/data/bsl_raw', exist_ok=True)

!kaggle datasets download -d eraakash/indian-sign-language-hand-landmarks-dataset -p /content/data/isl_raw --unzip
!kaggle datasets download -d alifsathar/bsl-fingerspelling-dataset -p /content/data/bsl_raw --unzip

print("\n--- ISL raw files (first few) ---")
for root, _, fs in os.walk('/content/data/isl_raw'):
    for f in fs[:5]:
        print(os.path.join(root, f))

print("\n--- BSL raw folder structure (first few) ---")
count = 0
for root, dirs, fs in os.walk('/content/data/bsl_raw'):
    print(root, '-> dirs:', dirs[:5], ' files:', fs[:3])
    count += 1
    if count > 8:
        break


## 4. Shared feature pipeline — must match `normalize.js` exactly

This is the most important part of the whole notebook. If this drifts even slightly from `src/utils/normalize.js`, the trained model will get different numbers in the browser than it saw during training, and predictions will be wrong or unstable — even if training accuracy looked great in Colab.


In [ ]:
import numpy as np

WRIST = 0
NUM_LANDMARKS = 21

def normalize_landmarks(pts):
    '''
    pts: (21, 3) array-like of a single hand's [x, y, z] landmarks.

    Mirrors src/utils/normalize.js -> normalizeLandmarks() EXACTLY:
      1. subtract the wrist (landmark 0) from every point
      2. divide every coordinate by the single largest absolute value
         across the whole hand (one uniform scale, not per-axis)

    Returns a flat (63,) float32 array:
      [x0,y0,z0, x1,y1,z1, ..., x20,y20,z20]
    '''
    pts = np.asarray(pts, dtype=np.float32).reshape(NUM_LANDMARKS, 3)
    wrist = pts[WRIST].copy()
    centered = pts - wrist
    max_val = float(np.max(np.abs(centered))) if centered.size else 0.0
    scaled = centered / max_val if max_val > 0 else centered
    return scaled.reshape(-1).astype(np.float32)


def build_two_hand_126(hand0, hand1):
    '''
    hand0, hand1: each either (21,3) landmarks or None (hand not present).

    Mirrors src/utils/normalize.js -> buildTwoHand126() EXACTLY:
      normalize each hand independently, then concatenate:
      [hand0_normalized(63), hand1_normalized(63)] = 126 values.
    A missing hand becomes 63 zeros — same fallback the JS uses.
    '''
    a = normalize_landmarks(hand0) if hand0 is not None else np.zeros(63, dtype=np.float32)
    b = normalize_landmarks(hand1) if hand1 is not None else np.zeros(63, dtype=np.float32)
    return np.concatenate([a, b]).astype(np.float32)


NUM_FEATURES_TWO_HAND = 126
print("Feature pipeline ready. Output shape per sample:", NUM_FEATURES_TWO_HAND)


### ⚠️ Hand ordering — and how this notebook handles it

MediaPipe doesn't guarantee "hand 0" is the left hand — it returns hands in whatever order it detected them in that particular frame, and `useMediaPipe.js` forwards them to `useSignModel` in that same detection order. So the browser might hand your model `[left, right]` on one frame and `[right, left]` on the next for the identical sign.

To make the trained model robust to that, for every genuinely two-handed training sample this notebook **also adds a duplicate row with hand0/hand1 swapped**. That teaches the model the two halves of the 126-vector are interchangeable, without requiring any change to `useMediaPipe.js` or `normalize.js`.


## 5. Build the ISL training set from the landmark CSV

Kaggle contributors don't all use identical column names, so this section auto-detects the label column and the numeric coordinate columns, and tells you clearly if something doesn't match the expected shape (63 columns = one hand per row, 126 = two hands per row) so you can adjust in one place.


In [ ]:
import pandas as pd
import glob

isl_csvs = glob.glob('/content/data/isl_raw/**/*.csv', recursive=True)
print("Found CSV file(s):", isl_csvs)
assert isl_csvs, "No CSV found under /content/data/isl_raw — check the dataset really contains landmark CSVs."

isl_df = pd.concat([pd.read_csv(f) for f in isl_csvs], ignore_index=True)
print("Combined shape:", isl_df.shape)
isl_df.head()


In [ ]:
def find_label_column(df):
    for cand in ['label', 'class', 'letter', 'sign', 'target', 'Label', 'Class']:
        if cand in df.columns:
            return cand
    non_numeric = [c for c in df.columns if df[c].dtype == object]
    if non_numeric:
        return non_numeric[-1]
    raise ValueError(
        "Couldn't auto-detect a label column. Run `print(isl_df.columns.tolist())`, "
        "find the right one yourself, and set LABEL_COL = 'your_column_name' manually."
    )

LABEL_COL = find_label_column(isl_df)
print("Detected label column:", LABEL_COL)

coord_cols = [c for c in isl_df.columns if c != LABEL_COL and pd.api.types.is_numeric_dtype(isl_df[c])]
n = len(coord_cols)
print(f"Detected {n} numeric coordinate columns")

if n == 63:
    print("→ This dataset stores ONE hand's 21×{x,y,z} per row.")
    HANDS_PER_ROW = 1
elif n == 126:
    print("→ This dataset stores TWO hands' 21×{x,y,z} per row.")
    HANDS_PER_ROW = 2
else:
    raise ValueError(
        f"Unexpected coordinate-column count ({n}), expected 63 or 126. "
        "Run `print(isl_df.columns.tolist())`, identify the real coordinate "
        "columns, and set `coord_cols` and `HANDS_PER_ROW` manually before "
        "re-running the next cell."
    )


In [ ]:
import string

KEEP_ONLY_LETTERS = True  # drop digits / non A-Z classes so ISL matches ASL's A-Z scope

def row_to_hand(flat63):
    return np.asarray(flat63, dtype=np.float32).reshape(21, 3)

coords = isl_df[coord_cols].to_numpy(dtype=np.float32)
labels = isl_df[LABEL_COL].astype(str).str.strip().str.upper().to_numpy()

X_isl, y_isl = [], []

for i in range(len(isl_df)):
    if HANDS_PER_ROW == 1:
        hand0, hand1 = row_to_hand(coords[i]), None
    else:
        h0 = row_to_hand(coords[i, :63])
        h1 = row_to_hand(coords[i, 63:126])
        # some datasets encode "hand not present" as all-zero — treat as missing
        hand1 = None if np.allclose(h1, 0) else h1
        hand0 = h0
        if np.allclose(hand0, 0) and hand1 is not None:
            hand0, hand1 = hand1, None

    X_isl.append(build_two_hand_126(hand0, hand1))
    y_isl.append(labels[i])

    if hand0 is not None and hand1 is not None:
        X_isl.append(build_two_hand_126(hand1, hand0))  # hand-order augmentation
        y_isl.append(labels[i])

X_isl = np.stack(X_isl)
y_isl = np.array(y_isl)

if KEEP_ONLY_LETTERS:
    mask = np.array([lbl in set(string.ascii_uppercase) for lbl in y_isl])
    X_isl, y_isl = X_isl[mask], y_isl[mask]

print("ISL dataset ready:", X_isl.shape, y_isl.shape)
print("Classes:", sorted(set(y_isl)))


## 6. Build the BSL training set from images (via MediaPipe Hands)

BSL fingerspelling has no ready-made landmark dataset yet, so we extract landmarks ourselves with `mediapipe.solutions.hands` in static-image mode, requesting up to 2 hands per image (matching BSL's two-handed alphabet).


In [ ]:
import glob

bsl_image_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png'):
    bsl_image_paths += glob.glob(f'/content/data/bsl_raw/**/{ext}', recursive=True)
print("Found", len(bsl_image_paths), "BSL images")

def label_from_path(path):
    # Assumes the common Kaggle layout: .../<LETTER>/image_0001.jpg
    return os.path.basename(os.path.dirname(path)).strip().upper()

detected_labels = sorted(set(label_from_path(p) for p in bsl_image_paths))
print("Labels detected from folder names:", detected_labels)
print(
    "\nIf these don't look like plain A-Z letters, open the folder tree printed in "
    "the download cell above and adjust `label_from_path` to match the real layout."
)


In [ ]:
# Optional: cap images per class to speed up a first run. Set to None to use everything.
MAX_IMAGES_PER_CLASS = None

if MAX_IMAGES_PER_CLASS:
    from collections import defaultdict
    by_class = defaultdict(list)
    for p in bsl_image_paths:
        by_class[label_from_path(p)].append(p)
    bsl_image_paths = [p for paths in by_class.values() for p in paths[:MAX_IMAGES_PER_CLASS]]
    print("Capped to", len(bsl_image_paths), "images total")


In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions
import string

KEEP_ONLY_LETTERS_BSL = True

# Download the hand landmarker model if not present
import os
MODEL_PATH = '/tmp/hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    !wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task -O {MODEL_PATH}
    print("Downloaded hand_landmarker.task")
else:
    print("hand_landmarker.task already exists")

# Create HandLandmarker using the new tasks API
base_options = mp.tasks.BaseOptions(model_asset_path=MODEL_PATH)
options = HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5
)

X_bsl, y_bsl = [], []
skipped = 0

with HandLandmarker.create_from_options(options) as landmarker:
    for idx, path in enumerate(bsl_image_paths):
        label = label_from_path(path)
        if KEEP_ONLY_LETTERS_BSL and label not in set(string.ascii_uppercase):
            continue

        img = cv2.imread(path)
        if img is None:
            skipped += 1
            continue

        # Convert BGR to RGB for MediaPipe
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        result = landmarker.detect(mp_image)
        
        if not result.hand_landmarks:
            skipped += 1
            continue

        hand_pts = [
            np.array([[lm.x, lm.y, lm.z] for lm in h], dtype=np.float32)
            for h in result.hand_landmarks
        ]
        hand0 = hand_pts[0]
        hand1 = hand_pts[1] if len(hand_pts) > 1 else None

        X_bsl.append(build_two_hand_126(hand0, hand1))
        y_bsl.append(label)

        if hand1 is not None:
            X_bsl.append(build_two_hand_126(hand1, hand0))  # hand-order augmentation
            y_bsl.append(label)

        if idx % 500 == 0:
            print(f"  processed {idx}/{len(bsl_image_paths)} images — {len(X_bsl)} samples so far, {skipped} skipped")

X_bsl = np.stack(X_bsl)
y_bsl = np.array(y_bsl)

print(f"\nDone. {X_bsl.shape[0]} samples built, {skipped} images skipped (no hand detected / unreadable).")
print("Classes:", sorted(set(y_bsl)))


## 7. Shared model architecture + training function

Both languages train the same way: a compact dense network on the 126-dim feature vector, saved with Keras `model.save()` so it converts cleanly to the **layers model** format `useSignModel.js` expects (`tf.loadLayersModel`) — identical in kind to how the shipped ASL model is packaged.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


def build_model(num_features, num_classes):
    model = models.Sequential([
        layers.Input(shape=(num_features,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


def train_language_model(X, y, lang_name, epochs=60, batch_size=64):
    print(f"\n{'='*60}\nTraining {lang_name.upper()}\n{'='*60}")

    encoder = LabelEncoder()
    y_int = encoder.fit_transform(y)
    class_names = list(encoder.classes_)
    num_classes = len(class_names)
    print(f"{num_classes} classes: {class_names}")

    X_train, X_val, y_train, y_val = train_test_split(
        X, y_int, test_size=0.15, random_state=42, stratify=y_int
    )

    model = build_model(X.shape[1], num_classes)
    model.summary()

    early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)
    reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop, reduce_lr],
        verbose=2,
    )

    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"\n{lang_name.upper()} validation accuracy: {val_acc:.4f}")

    return model, encoder, class_names, val_acc


## 8. Train ISL

In [ ]:
isl_model, isl_encoder, isl_classes, isl_val_acc = train_language_model(X_isl, y_isl, "isl")


## 9. Train BSL

In [ ]:
bsl_model, bsl_encoder, bsl_classes, bsl_val_acc = train_language_model(X_bsl, y_bsl, "bsl")


## 10. Export to TF.js — identical file layout to `public/model/asl/`

Each language gets exactly the 4 files SignBridge's `useSignModel.js` already knows how to load:

- `model.json` + `group1-shard1of1.bin` (or more shards) — the TF.js layers model
- `label_encoder.json` — `{ "classes": [...] }`, output-index order
- `model_metadata.json` — same shape as the ASL one (`num_features`, `num_classes`, `class_names`, `test_accuracy`, etc.)


In [ ]:
import subprocess, json, datetime, os

def export_language(model, class_names, val_acc, lang_id, out_root='/content/export'):
    out_dir = f'{out_root}/{lang_id}'
    keras_dir = f'/content/keras_{lang_id}'
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(keras_dir, exist_ok=True)

    # 1. Save as a Keras model first
    keras_path = f'{keras_dir}/model.keras'
    model.save(keras_path)

    # 2. Convert to a TF.js LAYERS model — same format as public/model/asl/
    subprocess.run(
        ['tensorflowjs_converter', '--input_format=keras', keras_path, out_dir],
        check=True,
    )

    # 3. label_encoder.json — identical shape to the shipped ASL file
    with open(f'{out_dir}/label_encoder.json', 'w') as f:
        json.dump({'classes': class_names}, f)

    # 4. model_metadata.json — identical shape to the shipped ASL file
    metadata = {
        'model_version': '1.0',
        'num_features': 126,
        'num_classes': len(class_names),
        'class_names': class_names,
        'feature_description': {
            'landmarks_per_hand': 63,
            'hands': 2,
            'total': 126,
        },
        'normalization': {
            'method': 'wrist_origin_max_scale_per_hand',
            'description': (
                "each hand: subtract its own wrist, divide by that hand's max abs "
                "distance; a missing hand is 63 zeros"
            ),
        },
        'test_accuracy': float(val_acc),
        'input_shape': [126],
        'trained_on': datetime.date.today().isoformat(),
    }
    with open(f'{out_dir}/model_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"\n✅ {lang_id.upper()} exported to {out_dir}:")
    for fname in sorted(os.listdir(out_dir)):
        print("   ", fname)

    return out_dir


isl_out = export_language(isl_model, isl_classes, isl_val_acc, 'isl')
bsl_out = export_language(bsl_model, bsl_classes, bsl_val_acc, 'bsl')


In [ ]:
# Quick sanity checks before you trust the export
def validate_export(out_dir, expected_features, expected_classes):
    with open(f'{out_dir}/model.json') as f:
        model_json = json.load(f)
    assert 'modelTopology' in model_json, f"{out_dir}/model.json missing modelTopology — conversion likely failed"

    with open(f'{out_dir}/label_encoder.json') as f:
        enc = json.load(f)
    assert len(enc['classes']) == expected_classes, "label_encoder.json class count mismatch"

    with open(f'{out_dir}/model_metadata.json') as f:
        meta = json.load(f)
    assert meta['num_features'] == expected_features, "model_metadata.json feature count mismatch"

    print(f"✅ {out_dir} passes sanity checks — {len(enc['classes'])} classes, {meta['num_features']} features")

validate_export(isl_out, 126, len(isl_classes))
validate_export(bsl_out, 126, len(bsl_classes))

print(f"\nFor reference, the shipped ASL model reports test_accuracy = 0.9913 in its own model_metadata.json.")
print(f"ISL validation accuracy: {isl_val_acc:.4f}")
print(f"BSL validation accuracy: {bsl_val_acc:.4f}")


## 11. Package everything for download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/signbridge_isl_bsl_models', 'zip', '/content/export')
print("Created /content/signbridge_isl_bsl_models.zip containing isl/ and bsl/ folders")

files.download('/content/signbridge_isl_bsl_models.zip')


## 12. Integration into SignBridge — the only steps needed

1. **Unzip** `signbridge_isl_bsl_models.zip`. You'll get an `isl/` folder and a `bsl/` folder, each with `model.json`, one or more `group1-shard*.bin` files, `label_encoder.json`, and `model_metadata.json`.
2. **Copy their contents** into your repo, replacing the placeholder files:
   ```
   isl/*  →  signBridge/react-app/public/model/isl/
   bsl/*  →  signBridge/react-app/public/model/bsl/
   ```
   (delete `PLACEHOLDER.md` in each folder once the real files are in place)
3. **Flip one word per language** in `signBridge/react-app/src/config/languages.js`:
   ```js
   isl: {
     ...
     status: 'available',   // was 'comingSoon'
     ...
   },
   bsl: {
     ...
     status: 'available',   // was 'comingSoon'
     ...
   },
   ```
4. That's it. `useSignModel.js` already knows how to load any `status: 'available'` language, `normalize.js` already has the matching `two-hand-126` pipeline, and `useMediaPipe.js` + the app screens already forward both hands through when `numHands > 1`. **No other code changes are required, and `public/model/asl/` is never touched.**
5. Run `npm run dev` inside `react-app/`, open the language picker, and switch to ISL or BSL to try it.

### Known limitation to be upfront about

Neither public dataset used here includes dedicated **space / delete / "nothing"** gestures the way the ASL dataset did (ASL's 29 classes include `del`, `nothing`, `space` alongside A–Z). That means for ISL and BSL right now: **letter recognition works, but the on-screen word-builder's space/delete gestures won't trigger** for those two languages yet — `isSpace`/`isDel`/`isNothing` in `useSignModel.js`'s prediction object will simply never come back `true`, since those classes don't exist in `label_encoder.json`. To close that gap later, add labeled examples for those 3 states to the training data (e.g. record short clips of "hand at rest" for `nothing`, and pick two simple two-handed gestures for `space`/`del`) and retrain — no other change needed.

### If you want to swap datasets, add a language, or improve accuracy later

- **Different dataset**: change the `kaggle datasets download -d <slug>` line in section 3 and re-run from there — the parsing cells print diagnostics if the new layout doesn't match.
- **A 5th/6th language**: copy this notebook's pattern (download → landmarks → `build_two_hand_126` or a new pipeline → train → export), then register it the same way as ISL/BSL in `languages.js`.
- **Better accuracy**: raise `MAX_IMAGES_PER_CLASS` (or set it to `None`), add more `Dense` capacity, or add classic image augmentation (rotation/brightness) *before* the MediaPipe extraction step for the BSL images.
- **Never retrain ASL from this notebook** — if ASL ever needs retraining, do it as a separate effort against the `single-hand-78` pipeline in its own notebook, so this one stays purely additive.
